## Assignment 2 : Network Flow based Attack Classification
정보보호와시스템보안 과제 2 : 네트워크 플로우 Feature 기반 공격 유형 분류 (Multi-class Classification)

### Goal
- 과제 1에서 다룬 네트워크 플로우 Feature 15종으로 **정상 트래픽과 공격 유형 (총 16개 클래스)** 을 분류하는 모델 학습
- 제공된 `train.parquet`으로 학습하고 `valid.parquet`으로 검증하여, 높은 **Macro F1-score**를 얻는 것이 목표
- 최대 3인 1팀으로 진행합니다.

### Provided Data
| Name | Description |
| --- | --- |
| `train.parquet` | 학습 데이터 |
| `valid.parquet` | 검증 데이터 · 하이퍼파라미터 조정 및 성능 확인용 |
- `code.ipynb`가 위치한 디렉토리에 반드시 위 2개 파일을 두고 실행해주세요.
- 네트워크 식별자(5-Tuple: IP · Port · Protocol)와 `Timestamp`는 제거된 상태로 제공됩니다.
- **제공된 데이터 임의로 수정 및 변형 금지**
    - 수정 내용 확인 시, 0점 처리

### Features
| Column | Description | 과제 1과의 차이 |
| --- | --- | --- |
| `duration` | 플로우 지속 시간 | 단위 **마이크로초(μs)** |
| `fwd_pkts` / `bwd_pkts` | 방향별 패킷 수 | - |
| `fwd_bytes` / `bwd_bytes` | 방향별 전체 길이 | IP 전체 길이가 아닌 **Payload 길이** |
| `fwd_pkt_len_min` / `max` / `mean` / `std` | Forward 패킷 길이 통계 | Payload 길이 기준 · std는 **표본 표준편차 (n-1)** |
| `bwd_pkt_len_min` / `max` / `mean` / `std` | Backward 패킷 길이 통계 | Payload 길이 기준 · std는 **표본 표준편차 (n-1)** |
| `flow_iat_mean` / `flow_iat_std` | 패킷 간 시간 간격 통계 | 단위 **마이크로초(μs)** · std는 **표본 표준편차 (n-1)** |
- 데이터셋은 수정된 CICFlowMeter로 생성되어, 과제 1의 변환 결과와 이름은 같지만 계산 방식이 일부 다릅니다.
- 플로우 종료 조건도 과제 1(60초 무통신)과 달리 120초 Timeout + TCP FIN / RST 기준입니다.

### Classes
| Label | 설명 | Label | 설명 |
| --- | --- | --- | --- |
| `BENIGN` | 정상 트래픽 | `Portscan` | 외부 포트 스캔 |
| `DoS Hulk` | HTTP Flood | `Infiltration` | 내부 PC 감염 (Meterpreter) |
| `DoS GoldenEye` | HTTP Keep-Alive Flood | `Infiltration - Portscan` | 감염된 내부 PC에서의 포트 스캔 |
| `DoS Slowloris` | Slow HTTP Header | `FTP-Patator` | FTP Brute Force |
| `DoS Slowhttptest` | Slow HTTP Body | `SSH-Patator` | SSH Brute Force |
| `DDoS` | LOIC HTTP Flood | `Web Attack - Brute Force` | 웹 로그인 Brute Force |
| `Heartbleed` | OpenSSL Heartbleed | `Web Attack - XSS` | Cross-Site Scripting |
| `Botnet` | Ares Botnet | `Web Attack - SQL Injection` | SQL Injection |
- 라벨은 CIC-IDS-2017 원본 라벨을 그대로 사용합니다.
- `Heartbleed`, `Infiltration`, `Web Attack` 3종(총 5개 클래스)은 학습 데이터가 2~15개뿐인 극소수 클래스입니다.

### Rules
1. 학습 데이터
    - 제공된 `train.parquet`, `valid.parquet`만 사용 가능합니다.
    - CIC-IDS-2017 원본 등 **외부 데이터, 사전 학습된 모델 파일 사용 금지**
2. 데이터 전처리 및 모델
    - 데이터 전처리(`코드 셀 3`)와 모델 학습(`코드 셀 4`)은 **모두 자유롭게 구현**합니다.
        - Feature 변형 및 생성, Feature 선택, 정규화, 모델 종류(scikit-learn, LightGBM, XGBoost, PyTorch 등), 클래스 불균형 처리, 앙상블 등 모두 자유
    - 단, 아래 3가지는 정해진 형식을 지켜야 합니다. (`코드 셀 5`와 조교 평가에서 동일하게 사용)
        - `transform_column_func` : 원본 Feature로 모델 입력 컬럼을 만드는 변환식 (`{새 컬럼명: lambda x: ...}`)
        - `predict(X)` : 전처리를 거친 `X`의 각 행에 대한 **클래스 이름(`CLASSES` 중 하나) 배열**을 반환하는 함수
        - `model` : 학습된 최종 모델 (`코드 셀 5`에서 `output/model.joblib`으로 저장)
            - 정규화 등 학습된 객체가 여러 개라면 `Pipeline` 또는 `dict`로 묶어 `model`에 저장하세요.
    - 모든 평가 데이터는 원본 파일을 다시 읽어 **`predict(preprocess(X))`** 순서로 추론합니다.
    - 정규화 등 통계값(평균, 표준편차 등)은 학습 데이터(`train_X`)에서만 계산해야 합니다.
    - 오버샘플링 / 언더샘플링 등 행 개수를 바꾸는 처리는 `코드 셀 4`에서 학습 데이터에 대해서만 수행하세요.
    - `valid_X`, `valid_y`는 하이퍼파라미터 조정 및 Early Stopping 용도로만 사용하며, 학습 데이터에 합치는 것을 금지합니다.
3. 재현성
    - 모든 무작위성은 `SEED = 2026`으로 고정해야 합니다.
        - `random` · `numpy` · `PyTorch` · `TensorFlow`는 `코드 셀 2`의 `set_seed(SEED)`로 고정되어 있습니다.
        - scikit-learn · LightGBM · XGBoost 등 모델과 분할 함수에는 `random_state=SEED` (또는 `seed=SEED`)를 반드시 지정하세요.
    - 조교가 제출된 코드를 **처음부터 다시 실행**하여 성능을 검증합니다.
    - 제출한 Valid 결과(`output/pred.csv`, `코드 셀 6` 출력)와 재현 결과가 크게 다르거나, 실행 오류가 발생할 경우 별도 소명을 요청합니다.
4. 평가
    - 평가 지표 : 클래스별 F1-score의 평균인 **Macro F1-score** (Accuracy 아님)
    - 배점 : 모델 성능 평가 결과 (기본 점수 3, 최대 점수 10) + 진행 보고서 (5)

### Importance
1. 제공된 코드의 수정 부분 외 임의 수정을 금지합니다.
    - 데이터 로드, 시드, 평가 함수 등 중요 변인 통제가 적용되어 있습니다.
    - `# TODO` ~ `# TODO 끝` 주석으로 감싸진 부분에서만 작업을 진행하세요.
    - 전체 `.ipynb` 파일을 AI에 제공하여 AI가 만든 결과로 과제를 제출할 경우, 그에 따른 불이익이 있습니다.
2. 제공된 데이터 임의 수정 및 변형을 금지합니다.
    - 수정 내용 확인 시, 0점 처리합니다.

### To-Do
1. 라이브러리 설치 및 임포트 (`코드 셀 1`)
    - 추가로 필요한 라이브러리는 셀 위쪽의 `# TODO: 라이브러리 설치` 부분에 `%pip install`로 설치하고, 아래쪽 `# TODO: 라이브러리 임포트` 부분에서 임포트
2. 데이터 전처리 (`코드 셀 3`, `# TODO: 빈칸 1`)
    - `transform_column_func`에 모델 입력으로 사용할 컬럼의 변환식 작성 (기본값 : 원본 Feature 15개 그대로 사용)
3. 모델 학습 (`코드 셀 4`, `# TODO: 빈칸 2`)
    - 정규화, 모델 선언 및 학습을 구현하고, 최종 모델을 `model`에 저장한 뒤 추론 함수 `predict(X)` 정의 (기본값 : StandardScaler + Logistic Regression)
4. 최종 모델 및 Valid 추론 결과 저장 (`코드 셀 5`)
    - `output/model.joblib` : 학습된 최종 모델
    - `output/pred.csv` : Valid 데이터 추론 결과
5. Valid 성능 확인 (`코드 셀 6`)
    - `output/pred.csv`로 계산한 Accuracy, 클래스별 Precision / Recall / F1-score, 최종 Macro F1-score 확인
6. 진행 보고서 작성 (자유 양식, pdf)
    - 데이터 분석, 모델 선택 및 개선 과정, 실험 결과 (성능 비교), 한계점 등을 정리
7. 제출용 zip 파일 생성
    - `code.ipynb` / `output/model.joblib` / `output/pred.csv` / `인공지능 사용 내역.pdf` / `진행 보고서.pdf` 5개 파일을 zip 파일로 압축
        - `code.ipynb`는 반드시 전체 실행 이후 저장 (Ctrl+S)하고 압축 파일로 포함해야 합니다.
        - 생성형 AI 사용 시 사용 내용 캡처본을 반드시 포함해야 하며, 미제출 시 AI를 사용하지 않은 것으로 간주합니다.
    - 파일명은 `정보보호와시스템보안_과제2_팀번호.zip` 으로 할 것
    - 제출 기한 : 11월 17일 오후 11:59까지 (가상대학 제출)

In [13]:
# 코드 셀 1 - 모듈 임포트

# TODO: 라이브러리 설치 - 추가로 필요한 라이브러리를 아래에 설치하세요. (예 : %pip install -q lightgbm)
# TODO 끝

import os, random, time
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
# TODO: 라이브러리 임포트 - 필요한 라이브러리를 아래에 임포트하세요.
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
# TODO 끝

In [14]:
# 코드 셀 2 - 변수 설정 (Seed, 입력, 출력 위치) 및 기본 함수 선언
SEED = 2026

TRAIN_PARQUET = "output/train.parquet"
VALID_PARQUET = "output/valid.parquet"
OUTPUT_DIR = "output"
MODEL_FILE = os.path.join(OUTPUT_DIR, "model.joblib")
PRED_CSV = os.path.join(OUTPUT_DIR, "pred.csv")

LABEL_COL = "Label"
FEATURES = [
    "duration", "fwd_pkts", "bwd_pkts", "fwd_bytes", "bwd_bytes",
    "fwd_pkt_len_min", "fwd_pkt_len_max", "fwd_pkt_len_mean", "fwd_pkt_len_std",
    "bwd_pkt_len_min", "bwd_pkt_len_max", "bwd_pkt_len_mean", "bwd_pkt_len_std",
    "flow_iat_mean", "flow_iat_std",
]
CLASSES = [
    "BENIGN", "DoS Hulk", "DoS GoldenEye", "DoS Slowloris", "DoS Slowhttptest", "DDoS", "Heartbleed",
    "Portscan", "Infiltration", "Infiltration - Portscan", "FTP-Patator", "SSH-Patator",
    "Web Attack - Brute Force", "Web Attack - XSS", "Web Attack - SQL Injection", "Botnet",
]

def set_seed(seed):
    # Python / NumPy
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    # PyTorch (설치된 경우)
    try:
        import torch
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass
    # TensorFlow / Keras (설치된 경우)
    try:
        import tensorflow as tf
        os.environ["TF_DETERMINISTIC_OPS"] = "1"
        tf.random.set_seed(seed)
        tf.keras.utils.set_random_seed(seed)
    except ImportError:
        pass

def analysis_data(name, y):
    counts = pd.Series(y).value_counts()
    print("=" * 80)
    print(f"Filename       : {name}")
    print("=" * 80)
    print(f"Total Flows    : {len(y):,}")
    for label in CLASSES:
        n = int(counts.get(label, 0))
        print(f"  {label:<28} {n:>10,} ({n / len(y):7.3%})")

def check_pred(y_true, y_pred):
    y_pred = np.asarray(y_pred).astype(str).ravel()
    unknown = sorted(set(y_pred) - set(CLASSES))
    assert len(y_pred) == len(y_true), f"예측 개수 {len(y_pred):,}와 데이터 개수 {len(y_true):,}가 다릅니다."
    assert not unknown, f"predict는 CLASSES의 클래스 이름을 반환해야 합니다. (잘못된 값 : {unknown[:5]})"
    return y_pred

def evaluate(name, y_true, y_pred):
    y_pred = check_pred(y_true, y_pred)
    labels = [label for label in CLASSES if label in set(y_true)]  # 평가 데이터에 존재하는 클래스
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    result = {"accuracy": accuracy_score(y_true, y_pred),
              "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
              "per_class": {label: report[label] for label in labels}}
    print("=" * 80)
    print(f"Evaluation     : {name}")
    print("=" * 80)
    print(f"Accuracy       : {result['accuracy']:.6f}")
    print("-" * 80)
    print(f"  {'Label':<28} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    for label, r in result["per_class"].items():
        print(f"  {label:<28} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f} {int(r['support']):>10,}")
    print("-" * 80)
    print(f"Macro F1       : {result['macro_f1']:.6f}")
    return result

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [15]:
# 코드 셀 3 - 데이터 전처리
train_df = pd.read_parquet(TRAIN_PARQUET)
valid_df = pd.read_parquet(VALID_PARQUET)

# TODO: 빈칸 1 - 모델 입력으로 사용할 컬럼의 변환식을 작성하세요. (기본값 : 원본 Feature 15개 그대로 사용)
# HINT: {새 컬럼명: lambda x: ...} 형식이며, x는 FEATURES 15개 컬럼의 원본 DataFrame입니다. (Label 사용 불가)
# HINT: Feature 변형 / 생성 (예 : 'bytes_ratio': lambda x: x['bwd_bytes'] / (x['fwd_bytes'] + 1)), Feature 선택 등을 자유롭게 구현할 수 있습니다.
# HINT: 변환식은 Train / Valid / 평가 데이터에 동일하게 적용되므로, 행 개수를 바꾸면 안 됩니다.
transform_column_func = {
    'duration' : lambda x: x['duration'],
    'fwd_pkts' : lambda x: x['fwd_pkts'],
    'bwd_pkts' : lambda x: x['bwd_pkts'],
    'fwd_bytes' : lambda x: x['fwd_bytes'],
    'bwd_bytes' : lambda x: x['bwd_bytes'],
    'fwd_pkt_len_min' : lambda x: x['fwd_pkt_len_min'],
    'fwd_pkt_len_max' : lambda x: x['fwd_pkt_len_max'],
    'fwd_pkt_len_mean' : lambda x: x['fwd_pkt_len_mean'],
    'fwd_pkt_len_std' : lambda x: x['fwd_pkt_len_std'],
    'bwd_pkt_len_min' : lambda x: x['bwd_pkt_len_min'],
    'bwd_pkt_len_max' : lambda x: x['bwd_pkt_len_max'],
    'bwd_pkt_len_mean' : lambda x: x['bwd_pkt_len_mean'],
    'bwd_pkt_len_std' : lambda x: x['bwd_pkt_len_std'],
    'flow_iat_mean' : lambda x: x['flow_iat_mean'],
    'flow_iat_std' : lambda x: x['flow_iat_std'],
    'pkts_spd' : lambda x: x['fwd_pkts'] / (x['duration'] / 1e6 + 1e-6),
    'bytes_spd' : lambda x: x['fwd_bytes'] / (x['duration'] / 1e6 + 1e-6),
    'fwd_pkts_ratio' : lambda x: x['fwd_pkts'] / (x['fwd_pkts'] + x['bwd_pkts'] + 1e-6),
    'iat_cv' : lambda x: x['flow_iat_std'] / (x['flow_iat_mean'] + 1e-6),
    'pkt_len_speed' : lambda x: x['fwd_pkt_len_mean'] / (x['duration'] / 1e6 + 1e-6),
    'fwd_len_dis' : lambda x: x['fwd_pkt_len_max'] - (x['fwd_pkt_len_mean']), 
    'fwd_len_ratio' : lambda x: x['fwd_pkt_len_std'] / (x['fwd_pkt_len_mean'] + 1e-6)
}
# TODO: 빈칸 1 끝

def preprocess(df):
    return pd.DataFrame({new_col: func(df) for new_col, func in transform_column_func.items()})

train_X, train_y = preprocess(train_df[FEATURES]), train_df[LABEL_COL].to_numpy()
valid_X, valid_y = preprocess(valid_df[FEATURES]), valid_df[LABEL_COL].to_numpy()
analysis_data(TRAIN_PARQUET, train_y)
analysis_data(VALID_PARQUET, valid_y)
print(f"Train          : {train_df[FEATURES].shape} -> {train_X.shape}")
print(f"Valid          : {valid_df[FEATURES].shape} -> {valid_X.shape}")
print("Preprocessing Complete!")

Filename       : output/train.parquet
Total Flows    : 338,444
  BENIGN                          282,328 (83.419%)
  DoS Hulk                         31,694 ( 9.365%)
  DoS GoldenEye                     1,514 ( 0.447%)
  DoS Slowloris                       691 ( 0.204%)
  DoS Slowhttptest                    346 ( 0.102%)
  DDoS                             19,029 ( 5.622%)
  Heartbleed                            2 ( 0.001%)
  Portscan                            287 ( 0.085%)
  Infiltration                          7 ( 0.002%)
  Infiltration - Portscan             991 ( 0.293%)
  FTP-Patator                         794 ( 0.235%)
  SSH-Patator                         592 ( 0.175%)
  Web Attack - Brute Force             15 ( 0.004%)
  Web Attack - XSS                      4 ( 0.001%)
  Web Attack - SQL Injection            3 ( 0.001%)
  Botnet                              147 ( 0.043%)
Filename       : output/valid.parquet
Total Flows    : 169,220
  BENIGN                          141,164 

In [16]:
# 코드 셀 4 - 모델 학습 (XGBoost 단독 분석)
set_seed(SEED)
start = time.time()

# TODO: 빈칸 2 - 모델을 선언 및 학습하고, 추론 함수 predict(X)를 정의하세요. (기본값 : StandardScaler + Logistic Regression)
# HINT: train_X / valid_X 는 전처리를 거친 DataFrame, train_y / valid_y 는 클래스 이름(문자열) 배열입니다.
# HINT: 학습된 최종 모델은 반드시 model 변수에 저장해야 합니다. (정규화 등 학습된 객체가 여러 개라면 Pipeline 또는 dict로 묶기)
# HINT: predict(X)는 반드시 정의해야 하며, 전처리를 거친 X의 각 행에 대한 클래스 이름(CLASSES 중 하나) 배열을 반환해야 합니다.
# HINT: 무작위성이 있는 모든 모델에는 random_state=SEED (또는 seed=SEED)를 지정해야 합니다.
# HINT: valid_X, valid_y는 하이퍼파라미터 조정 및 Early Stopping 용도로만 사용하세요.

le = LabelEncoder()
train_y_enc = le.fit_transform(train_y)

unique, counts = np.unique(train_y_enc, return_counts=True)
sampling_strategy = {cls: max(cnt, 50) for cls, cnt in zip(unique, counts)}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=SEED)
train_X_res, train_y_res = ros.fit_resample(train_X, train_y_enc)

print("[1/1] XGBoost 단독 학습 시작...")
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    eval_metric='mlogloss'
)
xgb.fit(train_X_res, train_y_res)
print(f" -> XGBoost 학습 완료 ({time.time() - t0:.2f}s)")

# 피처 중요도 출력 (분석용)
feat_imp = pd.Series(xgb.feature_importances_, index=train_X.columns).sort_values(ascending=False)
print("\n[XGBoost Top 10 Feature Importances]")
print(feat_imp.head(10))

model = {
    'classifier': xgb,
    'label_encoder': le
}

def predict(X):
    pred_enc = model['classifier'].predict(X)
    return model['label_encoder'].inverse_transform(pred_enc)
# TODO: 빈칸 2 끝

[1/1] XGBoost 단독 학습 시작...
 -> XGBoost 학습 완료 (35.37s)

[XGBoost Top 10 Feature Importances]
bwd_pkt_len_std     0.446324
bwd_pkt_len_mean    0.131130
fwd_bytes           0.127289
fwd_pkt_len_mean    0.054911
fwd_pkt_len_std     0.049893
bwd_pkt_len_max     0.031291
fwd_pkt_len_max     0.029076
bwd_bytes           0.028673
bytes_spd           0.024245
fwd_len_dis         0.013237
dtype: float32


In [17]:
# 코드 셀 5 - 최종 모델 및 Valid 추론 결과 저장
assert callable(globals().get("predict")), "코드 셀 4에서 predict(X) 함수를 정의해야 합니다."
assert globals().get("model") is not None, "코드 셀 4에서 학습된 최종 모델을 model 변수에 저장해야 합니다."
eval_df = pd.read_parquet(VALID_PARQUET)  # 원본 파일을 다시 읽어 추론 (평가 데이터와 동일한 방식)
y_pred = check_pred(eval_df[LABEL_COL], predict(preprocess(eval_df[FEATURES])))

pd.DataFrame({"label": eval_df[LABEL_COL], "pred": y_pred}).to_csv(PRED_CSV, index_label="row")
try:
    joblib.dump(model, MODEL_FILE, compress=3)
except Exception as e:
    raise RuntimeError(f"model 저장 실패 - lambda 등 저장할 수 없는 객체가 포함되어 있는지 확인하세요. ({type(e).__name__}: {e})")

print("=" * 80)
for path in [MODEL_FILE, PRED_CSV]:
    size = os.path.getsize(path) / 1024
    print(f"Saved          : {path} ({size / 1024:,.2f} MB)" if size >= 1024 else f"Saved          : {path} ({size:,.1f} KB)")

Saved          : output\model.joblib (1.54 MB)
Saved          : output\pred.csv (3.51 MB)


In [18]:
# 코드 셀 6 - Valid 성능 평가
pred_df = pd.read_csv(PRED_CSV, dtype=str, keep_default_na=False)
y_true = pd.read_parquet(VALID_PARQUET)[LABEL_COL].to_numpy()
result = evaluate(VALID_PARQUET, y_true, pred_df["pred"].to_numpy())
print("All Complete! :)")

Evaluation     : output/valid.parquet
Accuracy       : 0.999415
--------------------------------------------------------------------------------
  Label                         Precision     Recall         F1    Support
  BENIGN                           0.9997     0.9997     0.9997    141,164
  DoS Hulk                         0.9999     0.9999     0.9999     15,847
  DoS GoldenEye                    1.0000     0.9974     0.9987        757
  DoS Slowloris                    1.0000     1.0000     1.0000        345
  DoS Slowhttptest                 0.9885     0.9942     0.9914        173
  DDoS                             1.0000     1.0000     1.0000      9,514
  Heartbleed                       1.0000     1.0000     1.0000          1
  Portscan                         0.8456     0.8811     0.8630        143
  Infiltration                     0.0000     0.0000     0.0000          4
  Infiltration - Portscan          0.9438     0.9495     0.9466        495
  FTP-Patator                 